# 03 — Per-Image PCA Clustering (Variance-Threshold PCA)

Same pipeline as before, but instead of fixing **10 components per image type**,
PCA is fitted separately for each image type and keeps however many components
are needed to explain **≥59.2 % of variance** — matching the global PCA baseline.

The 5 per-image reduced vectors are then concatenated into one per-patient
feature vector (variable total length), and clustered with
**Robust Mahalanobis distance + FasterPAM k-medoids (k=4)**.

**Outputs**
- `pca_per_image_features.csv` — 416 rows × (1 + total_features) columns
- `pca_per_image_clustering_results.csv` — `patient_id`, `cluster`, `CDR`

In [1]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "robust-mixed-dist", "kmedoids", "scikit-learn", "-q"],
    check=True
)

CompletedProcess(args=['C:\\Users\\aregk\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', 'robust-mixed-dist', 'kmedoids', 'scikit-learn', '-q'], returncode=0)

In [2]:
import glob
import os

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from skimage.transform import resize
from sklearn.decomposition import PCA

from robust_mixed_dist.mixed import S_robust
from robust_mixed_dist.quantitative import robust_mahalanobis_dist_matrix
import kmedoids

## Step 1 — Parameters and image-type patterns

`VARIANCE_TARGET = 0.592` tells sklearn's PCA to select the **minimum number of
components** whose cumulative explained variance reaches 59.2 %.
This threshold matches the global PCA baseline so the total information
retained per image type is comparable across methods.

In [3]:
DATA_DIR     = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
CSV_SOURCE   = os.path.join(DATA_DIR, "method_1_HOG_PCA", "oasis_combined.csv")
OUT_FEATURES = os.path.join(DATA_DIR, "notebooks", "pca_per_image_features.csv")
OUT_CLUSTERS = os.path.join(DATA_DIR, "notebooks", "pca_per_image_clustering_results.csv")

# HOG parameters (same across all notebooks)
TARGET_SIZE      = (128, 128)
ORIENTATIONS     = 9
PIXELS_PER_CELL  = (8, 8)
CELLS_PER_BLOCK  = (2, 2)

VARIANCE_TARGET = 0.592   # keep components until cumulative variance >= 59.2%
K               = 4       # number of clusters

IMAGE_TYPES = {
    "cor":        "*_t88_gfc_cor_*.gif",
    "gfc_sag":    "*_t88_gfc_sag_*.gif",
    "gfc_tra":    "*_t88_gfc_tra_*.gif",
    "masked_tra": "*_t88_masked_gfc_tra_*.gif",
    "sbj_sag":    "*_sbj_111_sag_*.gif",
}

patient_dirs = sorted(
    d for d in glob.glob(os.path.join(DATA_DIR, "OAS1_*"))
    if os.path.isdir(d)
)
patient_ids = [os.path.basename(d) for d in patient_dirs]
N = len(patient_ids)
print(f"Found {N} patient folders")

Found 416 patient folders


## Step 2 — Helper functions

In [4]:
def load_image(path):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0
    return resize(arr, TARGET_SIZE, anti_aliasing=True)


def extract_hog(arr):
    features, _ = hog(
        arr,
        orientations=ORIENTATIONS,
        pixels_per_cell=PIXELS_PER_CELL,
        cells_per_block=CELLS_PER_BLOCK,
        visualize=True,
        feature_vector=True,
    )
    return features

## Step 3 — Extract HOG and fit per-image variance-threshold PCA

Passing a float (0 < value < 1) to `PCA(n_components=...)` instructs sklearn
to automatically select the minimum number of components whose cumulative
explained variance meets or exceeds that fraction.  Each image type gets its
own independent PCA, so the number of components can differ across types.

In [5]:
reduced_blocks = []
valid_mask     = np.ones(N, dtype=bool)
component_summary = {}

for type_key, pattern in IMAGE_TYPES.items():
    hog_rows  = []
    found_idx = []

    for i, p_dir in enumerate(patient_dirs):
        matches = glob.glob(os.path.join(p_dir, pattern))
        if not matches:
            print(f"  WARNING [{type_key}]: no match in {patient_ids[i]}")
            valid_mask[i] = False
            continue
        hog_rows.append(extract_hog(load_image(matches[0])))
        found_idx.append(i)

    hog_matrix = np.array(hog_rows, dtype=np.float32)   # (n_found, 8100)

    # Float n_components = select minimum components to reach VARIANCE_TARGET
    pca     = PCA(n_components=VARIANCE_TARGET, random_state=42)
    reduced = pca.fit_transform(hog_matrix)              # (n_found, n_selected)

    n_comp  = pca.n_components_
    cumvar  = pca.explained_variance_ratio_.sum()
    component_summary[type_key] = {"n_components": n_comp, "variance_explained": round(cumvar * 100, 2)}

    # Place results back into a full-N array (NaN rows for any missing patients)
    block = np.full((N, n_comp), np.nan, dtype=np.float32)
    for row, orig_i in enumerate(found_idx):
        block[orig_i] = reduced[row]
    reduced_blocks.append((type_key, block))

print(f"Patients present in all 5 types: {valid_mask.sum()} / {N}")

Patients present in all 5 types: 416 / 416


## Step 4 — Components selected per image type

Show how many PCA components each image type required to reach 59.2 % variance,
and the resulting total feature vector length per patient.

In [6]:
summary_df = pd.DataFrame(component_summary).T
summary_df.index.name = "image_type"
total_features = int(summary_df["n_components"].sum())

print(summary_df.to_string())
print(f"\nTotal features per patient : {total_features}")
print(f"(5 image types, variance threshold = {VARIANCE_TARGET*100:.1f}%)")

            n_components  variance_explained
image_type                                  
cor                110.0           59.209999
gfc_sag            105.0           59.259998
gfc_tra            107.0           59.330002
masked_tra         112.0           59.450001
sbj_sag             42.0           59.250000

Total features per patient : 476
(5 image types, variance threshold = 59.2%)


## Step 5 — Concatenate into one feature vector per patient

In [7]:
X_full = np.concatenate([block for _, block in reduced_blocks], axis=1)  # (N, total)

X_clean   = X_full[valid_mask]
ids_clean = [pid for pid, ok in zip(patient_ids, valid_mask) if ok]

print(f"Full matrix shape   : {X_full.shape}")
print(f"After dropping NaNs : {X_clean.shape}")

col_names = [
    f"{type_key}_{i+1}"
    for type_key, block in reduced_blocks
    for i in range(block.shape[1])
]
df_feat = pd.DataFrame(X_clean, columns=col_names)
df_feat.insert(0, "patient_id", ids_clean)

df_feat.to_csv(OUT_FEATURES, index=False)
print(f"\nSaved → {OUT_FEATURES}")
print(f"Shape  : {df_feat.shape}  (rows=patients, cols=patient_id + {total_features} PCA features)")
df_feat.iloc[:3, :8]

Full matrix shape   : (416, 476)
After dropping NaNs : (416, 476)

Saved → C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\pca_per_image_features.csv
Shape  : (416, 477)  (rows=patients, cols=patient_id + 476 PCA features)


,patient_id,cor_1,cor_2,cor_3,cor_4,cor_5,cor_6,cor_7
0,OAS1_0001,-2.733610,-0.262093,-0.753382,-0.805661,0.235610,0.087610,1.303593
1,OAS1_0002,1.980449,0.446797,-1.209317,0.267067,0.040793,-0.450332,0.901328
2,OAS1_0003,-1.515049,-1.209056,-0.827742,1.025612,0.436375,1.150286,0.239256


## Step 6 — Robust Mahalanobis distance matrix

All features are continuous PCA components.  The robust covariance matrix
is estimated with a 5 % trimming level before computing pairwise distances.

In [8]:
X_arr = X_clean.astype(float)

S_cov = S_robust(X_arr, method="trimmed", alpha=0.05)
print(f"Robust covariance matrix shape: {S_cov.shape}")

D = robust_mahalanobis_dist_matrix(X_arr, S_cov)

print(f"Distance matrix shape : {D.shape}")
print(f"Value range           : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric          : {np.allclose(D, D.T)}")

Robust covariance matrix shape: (476, 476)
Distance matrix shape : (416, 416)
Value range           : [0.0000, 35.8217]
Is symmetric          : True


## Step 7 — K-medoids clustering (FasterPAM, k=4)

In [9]:
result = kmedoids.fasterpam(D, medoids=K, random_state=42)

labels = np.array(result.labels)

print(f"Loss (sum of distances to medoids): {result.loss:.4f}")
print(f"Medoid patient IDs : {[ids_clean[i] for i in result.medoids]}")
print()
sizes = pd.Series(labels).value_counts().sort_index().rename("count")
print("Cluster sizes:")
print(sizes.to_string())

Loss (sum of distances to medoids): 10769.2252
Medoid patient IDs : ['OAS1_0108', 'OAS1_0054', 'OAS1_0262', 'OAS1_0131']

Cluster sizes:
0    101
1    142
2     94
3     79


## Step 8 — Evaluation: cluster vs CDR

In [10]:
cdr = pd.read_csv(CSV_SOURCE)[["patient_id", "CDR"]]

results = pd.DataFrame({"patient_id": ids_clean, "cluster": labels})
results = results.merge(cdr, on="patient_id", how="left")

print(f"Patients with CDR    : {results['CDR'].notnull().sum()}")
print(f"Patients without CDR : {results['CDR'].isnull().sum()}")
print()

has_cdr = results[results["CDR"].notnull()].copy()
has_cdr["CDR"] = has_cdr["CDR"].astype(str)

ct = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    margins=True,
    margins_name="Total"
)
print("Cluster × CDR (counts):")
ct

Patients with CDR    : 235
Patients without CDR : 181

Cluster × CDR (counts):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,25,21,8,2,56
1,47,20,10,0,77
2,35,18,6,0,59
3,28,11,4,0,43
Total,135,70,28,2,235


In [11]:
ct_norm = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    normalize="index"
).round(3)

print("Row-normalised (CDR proportion within each cluster):")
ct_norm

Row-normalised (CDR proportion within each cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.446,0.375,0.143,0.036
1,0.610,0.260,0.130,0.000
2,0.593,0.305,0.102,0.000
3,0.651,0.256,0.093,0.000


## Step 9 — Save results

In [12]:
results.to_csv(OUT_CLUSTERS, index=False)

print(f"Saved : {OUT_CLUSTERS}")
print(f"Shape : {results.shape}")
print()
print(results.head(5).to_string(index=False))

Saved : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\pca_per_image_clustering_results.csv
Shape : (416, 3)

patient_id  cluster  CDR
 OAS1_0001        1  0.0
 OAS1_0002        1  0.0
 OAS1_0003        2  0.5
 OAS1_0004        1  NaN
 OAS1_0005        0  NaN


---

## Experiment A — Euclidean Distance

Everything is identical to the method above (476 PCA components, same `X_clean`,
same k=4 FasterPAM) — only the distance metric changes.

Euclidean distance treats all 476 components as equally weighted axes in a flat
space, with no covariance correction.  Comparing this to Robust Mahalanobis
shows how much the choice of distance metric affects the clustering.

In [13]:
from sklearn.metrics.pairwise import euclidean_distances

D_euc = euclidean_distances(X_clean).astype(np.float64)

print(f"Euclidean distance matrix shape : {D_euc.shape}")
print(f"Value range                     : [{D_euc.min():.4f}, {D_euc.max():.4f}]")

Euclidean distance matrix shape : (416, 416)
Value range                     : [0.0000, 21.4906]


In [14]:
result_euc = kmedoids.fasterpam(D_euc, medoids=K, random_state=42)

labels_euc = np.array(result_euc.labels)

print(f"Loss (sum of distances to medoids): {result_euc.loss:.4f}")
print(f"Medoid patient IDs : {[ids_clean[i] for i in result_euc.medoids]}")
print()
sizes_euc = pd.Series(labels_euc).value_counts().sort_index().rename("count")
print("Cluster sizes:")
print(sizes_euc.to_string())

Loss (sum of distances to medoids): 5777.6217
Medoid patient IDs : ['OAS1_0016', 'OAS1_0141', 'OAS1_0366', 'OAS1_0294']

Cluster sizes:
0    101
1     89
2    107
3    119


In [15]:
results_euc = pd.DataFrame({"patient_id": ids_clean, "cluster": labels_euc})
results_euc = results_euc.merge(cdr, on="patient_id", how="left")

has_cdr_euc = results_euc[results_euc["CDR"].notnull()].copy()
has_cdr_euc["CDR"] = has_cdr_euc["CDR"].astype(str)

ct_euc = pd.crosstab(
    has_cdr_euc["cluster"],
    has_cdr_euc["CDR"],
    margins=True,
    margins_name="Total"
)
print("Cluster × CDR (counts) — Euclidean:")
ct_euc

Cluster × CDR (counts) — Euclidean:


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,41,37,17,0,95
1,24,10,3,0,37
2,34,17,7,2,60
3,36,6,1,0,43
Total,135,70,28,2,235


In [16]:
ct_euc_norm = pd.crosstab(
    has_cdr_euc["cluster"],
    has_cdr_euc["CDR"],
    normalize="index"
).round(3)

print("Row-normalised (CDR proportion within each cluster) — Euclidean:")
ct_euc_norm

Row-normalised (CDR proportion within each cluster) — Euclidean:


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.432,0.389,0.179,0.000
1,0.649,0.270,0.081,0.000
2,0.567,0.283,0.117,0.033
3,0.837,0.140,0.023,0.000


---

## Experiment B — Two-Stage PCA

A second global PCA is applied on top of the 476 per-image components,
compressing them down to **50 components**.  This removes remaining
cross-image redundancy and puts the feature space in the same dimensionality
as the original global-PCA baseline (Method 1).

The 50-component representation is then clustered twice:
- **B1** — Euclidean distance
- **B2** — Robust Mahalanobis distance

In [17]:
N_SECOND_PCA = 50

pca2     = PCA(n_components=N_SECOND_PCA, random_state=42)
X_stage2 = pca2.fit_transform(X_clean)   # (416, 50)

cumvar2 = np.cumsum(pca2.explained_variance_ratio_)
print(f"Second-stage PCA: {X_clean.shape[1]} → {N_SECOND_PCA} components")
print(f"Variance explained by 50 components: {cumvar2[-1]*100:.2f}%")
print(f"Output shape: {X_stage2.shape}")

Second-stage PCA: 476 → 50 components
Variance explained by 50 components: 58.66%
Output shape: (416, 50)


### B1 — Euclidean distance

In [18]:
D_b1 = euclidean_distances(X_stage2).astype(np.float64)

result_b1 = kmedoids.fasterpam(D_b1, medoids=K, random_state=42)
labels_b1 = np.array(result_b1.labels)

print(f"Loss: {result_b1.loss:.4f}")
print(f"Medoid patient IDs: {[ids_clean[i] for i in result_b1.medoids]}")
print()
print("Cluster sizes:")
print(pd.Series(labels_b1).value_counts().sort_index().rename("count").to_string())

results_b1 = pd.DataFrame({"patient_id": ids_clean, "cluster": labels_b1})
results_b1 = results_b1.merge(cdr, on="patient_id", how="left")
has_cdr_b1 = results_b1[results_b1["CDR"].notnull()].copy()
has_cdr_b1["CDR"] = has_cdr_b1["CDR"].astype(str)

ct_b1 = pd.crosstab(has_cdr_b1["cluster"], has_cdr_b1["CDR"],
                    margins=True, margins_name="Total")
print("\nCluster × CDR (counts) — B1 Euclidean:")
ct_b1

Loss: 3867.1086
Medoid patient IDs: ['OAS1_0268', 'OAS1_0383', 'OAS1_0012', 'OAS1_0294']

Cluster sizes:
0     85
1    111
2    119
3    101

Cluster × CDR (counts) — B1 Euclidean:


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,38,26,17,1,82
1,40,20,5,0,65
2,37,20,5,1,63
3,20,4,1,0,25
Total,135,70,28,2,235


In [19]:
ct_b1_norm = pd.crosstab(has_cdr_b1["cluster"], has_cdr_b1["CDR"],
                         normalize="index").round(3)
print("Row-normalised — B1 Euclidean:")
ct_b1_norm

Row-normalised — B1 Euclidean:


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.463,0.317,0.207,0.012
1,0.615,0.308,0.077,0.000
2,0.587,0.317,0.079,0.016
3,0.800,0.160,0.040,0.000


### B2 — Robust Mahalanobis distance

In [20]:
S_cov2 = S_robust(X_stage2.astype(float), method="trimmed", alpha=0.05)
D_b2   = robust_mahalanobis_dist_matrix(X_stage2.astype(float), S_cov2)

result_b2 = kmedoids.fasterpam(D_b2, medoids=K, random_state=42)
labels_b2 = np.array(result_b2.labels)

print(f"Robust covariance shape: {S_cov2.shape}")
print(f"Loss: {result_b2.loss:.4f}")
print(f"Medoid patient IDs: {[ids_clean[i] for i in result_b2.medoids]}")
print()
print("Cluster sizes:")
print(pd.Series(labels_b2).value_counts().sort_index().rename("count").to_string())

results_b2 = pd.DataFrame({"patient_id": ids_clean, "cluster": labels_b2})
results_b2 = results_b2.merge(cdr, on="patient_id", how="left")
has_cdr_b2 = results_b2[results_b2["CDR"].notnull()].copy()
has_cdr_b2["CDR"] = has_cdr_b2["CDR"].astype(str)

ct_b2 = pd.crosstab(has_cdr_b2["cluster"], has_cdr_b2["CDR"],
                    margins=True, margins_name="Total")
print("\nCluster × CDR (counts) — B2 Robust Mahalanobis:")
ct_b2

Robust covariance shape: (50, 50)
Loss: 3373.0713
Medoid patient IDs: ['OAS1_0409', 'OAS1_0132', 'OAS1_0383', 'OAS1_0294']

Cluster sizes:
0     86
1     68
2    122
3    140

Cluster × CDR (counts) — B2 Robust Mahalanobis:


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,26,17,6,0,49
1,16,14,3,0,33
2,42,25,10,1,78
3,51,14,9,1,75
Total,135,70,28,2,235


In [21]:
ct_b2_norm = pd.crosstab(has_cdr_b2["cluster"], has_cdr_b2["CDR"],
                         normalize="index").round(3)
print("Row-normalised — B2 Robust Mahalanobis:")
ct_b2_norm

Row-normalised — B2 Robust Mahalanobis:


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.531,0.347,0.122,0.000
1,0.485,0.424,0.091,0.000
2,0.538,0.321,0.128,0.013
3,0.680,0.187,0.120,0.013
